In [ ]:
import sys

sys.path.append("../src")
import pandas as pd
import geopandas as gpd
import glamos_processing as glamos
import meteoswiss_processing as meteo
import rioxarray
import xarray as xr

In [ ]:
gl = glamos.get_data(1982, 2025)
gl

In [ ]:
gl["id"].unique()

In [ ]:
poly = gpd.read_parquet(
    "../data/glacier_geometry_2013-2018.parquet",
)
poly

In [ ]:
poly = poly[poly["sgi-id"].isin(gl["id"].unique())]
poly

In [ ]:
gl["id"].unique()[~gl["id"].unique().isin(poly["sgi-id"])]

In [ ]:
df = pd.read_csv(
    "https://doi.glamos.ch/data/glacier_list/glacier_list.csv",
    skiprows=9,
    parse_dates=True,
    names=[
        "name",
        "id",
        "coordx",
        "coordy",
        "area",
        "survey_year",
        "length_change_data_available",
        "mass_balance_data_available",
        "volume_change_data_available",
    ],
)

In [ ]:
df[df.id.isin(gl["id"].unique()[~gl["id"].unique().isin(poly["sgi-id"])])]

In [ ]:
gl[gl.id.isin(gl["id"].unique()[~gl["id"].unique().isin(poly["sgi-id"])])]

Add "jupyter.kernels.trusted": [
        "/path/to/notebook"
    ], to VSCode settings.json and restart VSCode to make this work

In [ ]:
poly.explore()

In [ ]:
poly.boundary.explore()

In [ ]:
poly.bounds

In [ ]:
met = meteo.get_data(1982, 2025, True)
met

In [ ]:
met.sel(
    E=slice(2.792728e06, 2.793728e06),
    N=slice(1.182347e06, 1.182902e06),
    time="2020-01-01",
)["TabsM"].values

In [ ]:
poly

In [ ]:
poly.loc[46, "geometry"]

In [ ]:
met

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="angle from rectified to skew grid parameter lost in conversion to CF")

In [ ]:
met = met.rio.write_crs("EPSG:2056")

In [ ]:
poly.geometry.iloc[1]

In [ ]:
met.rio.clip([poly.geometry.iloc[1]])

In [ ]:
met.sel(E=gl.coordx.iloc[0], N=gl.coordy.iloc[0], method='nearest')

In [ ]:
met.sel(time=slice(gl.observation_start.iloc[0], gl.observation_end.iloc[0])).mean(
    dim=["N", "E"]
).isel(time=3)

In [ ]:
gl = gl.merge(poly, left_on="id", right_on="sgi-id", how="inner")
gl = gl.drop("sgi-id", axis=1)
gl

In [ ]:
def get_climate_features(row, climate: xr.Dataset):
    climate = climate.sel(time=slice(row['observation_start'], row['observation_end'])) # Select only relevant timeframe

    try:
        climate = climate.rio.clip([row['geometry']]) # Clip to glacier geometry
        climate = climate.mean(dim=['N', 'E']) # Calculate mean across grids
    except Exception as e:
        print(f'Failed to find climate grid in geometry for glacier id: {row['id']}, obs: {row['observation_start']}')
        print(e)
        print('Falling back to nearest coordinate')

        climate = climate.sel(E=row['coordx'], N=row['coordy'], method='nearest')

    q1 = climate.isel(time=0)
    q2 = climate.isel(time=1)
    q3 = climate.isel(time=2)
    q4 = climate.isel(time=3)

    return pd.Series({
        'q1h_temp': q1['TabsM'].values.item(), 
        'q2h_temp': q2['TabsM'].values.item(), 
        'q3h_temp': q3['TabsM'].values.item(), 
        'q4h_temp': q4['TabsM'].values.item(), 
        'q1h_prec': q1['RhiresM'].values.item(), 
        'q2h_prec': q2['RhiresM'].values.item(), 
        'q3h_prec': q3['RhiresM'].values.item(), 
        'q4h_prec': q4['RhiresM'].values.item()})
    

In [ ]:
gl.apply(get_climate_features, axis=1, args=(met,))